# APTOS 2019 — Entrenamiento y análisis exploratorio

**Proyecto Kaggle · Aplicaciones en Deep Learning · Universidad de Cundinamarca**
Yohan Nicolás Gómez Castañeda · Repositorio: `Nicolas2322/aptos-2019-blindness-detection`

---

### Antes de ejecutar

1. **Acelerador:** menú de la derecha → *Session options* → **Accelerator: GPU T4 x2** (o P100).
2. **Datos:** panel derecho → *Add Input* → busca `aptos2019-blindness-detection` → añade la **competencia**
   (no un dataset de otro usuario). La celda 2 localiza la ruta automáticamente, así que da igual si Kaggle
   la monta en `/kaggle/input/competitions/...` o en `/kaggle/input/...`.
3. **Internet:** *Session options* → **Internet: On** (hace falta para clonar el repositorio y descargar los pesos de ImageNet).
4. Menú **Run → Run all**.

### Qué produce

| Archivo | Contenido |
|---|---|
| `resultados.json` | QWK por pliegue, media, desviación y los cuatro umbrales |
| `eda_distribucion.png` | Distribución de clases |
| `eda_estadisticas.png` | Brillo y tamaño de las imágenes por grado |
| `eda_muestras.png` | Imágenes originales y preprocesadas por grado |
| `modelo_pliegue*.pt` | Pesos del mejor punto de cada pliegue |

Al terminar, descarga esos archivos desde la pestaña **Output** y adjúntalos en el chat.

## 1. Entorno

In [ ]:
import os, subprocess, sys, time
T0 = time.time()

REPO = "https://github.com/Nicolas2322/aptos-2019-blindness-detection.git"
if not os.path.exists("/kaggle/working/repo"):
    subprocess.run(["git", "clone", "--quiet", REPO, "/kaggle/working/repo"], check=True)

# albumentations reciente: el repositorio usa la API de RandomResizedCrop(size=...) y CoarseDropout(num_holes_range=...)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "albumentations>=1.4.21", "timm>=1.0.9", "mlflow>=2.14"], check=True)

sys.path.insert(0, "/kaggle/working/repo/src")
import torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO DISPONIBLE")
if not torch.cuda.is_available():
    print("\n*** Activa la GPU en Session options antes de continuar. ***")

## 2. Configuración

In [ ]:
import yaml
from pathlib import Path

def localizar_datos():
    """Kaggle monta las competencias en rutas distintas según la versión del entorno."""
    for p in [Path("/kaggle/input/competitions/aptos2019-blindness-detection"),
              Path("/kaggle/input/aptos2019-blindness-detection")]:
        if (p / "train.csv").exists():
            return p
    for csv in sorted(Path("/kaggle/input").glob("**/train.csv")):   # respaldo: búsqueda directa
        if (csv.parent / "train_images").is_dir():
            return csv.parent
    return None

DATOS = localizar_datos()
assert DATOS is not None, ("No se encontró train.csv bajo /kaggle/input. Añade la COMPETENCIA "
                           "'APTOS 2019 Blindness Detection' con 'Add Input'.")
SALIDA = Path("/kaggle/working"); SALIDA.mkdir(exist_ok=True)
print("Datos en:", DATOS)

cfg = yaml.safe_load(open("/kaggle/working/repo/configs/entrenamiento.yaml"))

# --- Ajusta estos dos valores según el tiempo disponible -----------------
PLIEGUES_A_ENTRENAR = [0]        # [0] ≈ 60–80 min. [0,1,2,3,4] ≈ 5–7 h (límite de sesión: 9 h)
cfg["epocas_ajuste"] = 25        # bájalo a 12 para una corrida rápida
# ------------------------------------------------------------------------

print({k: cfg[k] for k in ["arquitectura","tamano","lote","acumulacion","epocas_congeladas","epocas_ajuste","paciencia","pliegues","semilla"]})
print("Pliegues a entrenar:", PLIEGUES_A_ENTRENAR)

## 3. Análisis exploratorio

Corresponde a la actividad del cronograma prevista para el 14 de octubre. Produce las cifras y figuras
que sustentan el apartado de datos del informe final.

In [ ]:
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np, pandas as pd, cv2
plt.rcParams["font.family"] = "serif"

etiquetas = pd.read_csv(DATOS / "train.csv")
conteo = etiquetas.diagnosis.value_counts().sort_index()
NOMBRES = ["0\nSin RD", "1\nLeve", "2\nModerada", "3\nSevera", "4\nProliferativa"]

print(f"Total de imágenes etiquetadas: {len(etiquetas)}")
for g, n in conteo.items():
    print(f"  grado {g}: {n:5d}  ({100*n/len(etiquetas):5.1f} %)")

fig, ax = plt.subplots(figsize=(7, 3.6), dpi=160)
b = ax.bar(NOMBRES, conteo.values, color="#4a4a4a", width=0.6)
for r, v in zip(b, conteo.values):
    ax.text(r.get_x() + r.get_width()/2, v + 20, f"{v} ({100*v/len(etiquetas):.1f} %)", ha="center", fontsize=9)
ax.set_ylabel("Número de imágenes"); ax.set_xlabel("Grado de retinopatía diabética")
ax.set_ylim(0, conteo.max()*1.15); ax.spines[["top","right"]].set_visible(False)
plt.tight_layout(); plt.savefig(SALIDA/"eda_distribucion.png"); plt.show()

In [ ]:
# Estadísticas de imagen sobre una muestra estratificada (lectura reducida para que sea rápido)
from PIL import Image as PILImage

from aptos.preprocesamiento import estadisticas_imagen

muestra = (etiquetas.sample(frac=1, random_state=42)
           .groupby("diagnosis").head(60).reset_index(drop=True))

filas = []
for cid, g in zip(muestra.id_code, muestra.diagnosis):
    ruta = DATOS/"train_images"/f"{cid}.png"
    with PILImage.open(ruta) as im:          # dimensiones exactas sin decodificar la imagen
        ancho_real, alto_real = im.size
    img = cv2.cvtColor(cv2.imread(str(ruta), cv2.IMREAD_REDUCED_COLOR_4), cv2.COLOR_BGR2RGB)
    filas.append({**estadisticas_imagen(img), "alto": alto_real, "ancho": ancho_real, "diagnosis": g})
est = pd.DataFrame(filas)

print("Resolución original (muestra):")
print(est[["alto","ancho"]].describe().loc[["min","50%","max"]].round(0).to_string())
print(f"\nBrillo medio global: {est.brillo_medio.mean():.1f}   desviación entre imágenes: {est.brillo_medio.std():.1f}")
print(f"Fracción de retina media: {est.fraccion_retina.mean():.3f}")

fig, axes = plt.subplots(1, 2, figsize=(9, 3.4), dpi=160)
axes[0].boxplot([est[est.diagnosis==g].brillo_medio for g in range(5)], labels=list(range(5)),
                medianprops=dict(color="black"))
axes[0].set_xlabel("Grado"); axes[0].set_ylabel("Brillo medio de la retina")
axes[1].scatter(est.ancho, est.alto, s=10, alpha=0.5, color="#4a4a4a")
axes[1].set_xlabel("Ancho (px)"); axes[1].set_ylabel("Alto (px)")
for a in axes: a.spines[["top","right"]].set_visible(False)
plt.tight_layout(); plt.savefig(SALIDA/"eda_estadisticas.png"); plt.show()

In [ ]:
# Una imagen por grado, antes y después del preprocesamiento compartido
from aptos.preprocesamiento import preparar_imagen

fig, axes = plt.subplots(2, 5, figsize=(11, 4.6), dpi=160)
for g in range(5):
    cid = etiquetas[etiquetas.diagnosis == g].id_code.iloc[0]
    img = cv2.cvtColor(cv2.imread(str(DATOS/"train_images"/f"{cid}.png")), cv2.COLOR_BGR2RGB)
    _, proc = preparar_imagen(img)
    axes[0, g].imshow(img);  axes[0, g].set_title(f"Grado {g}", fontsize=10)
    axes[1, g].imshow(proc)
    for r in (0, 1): axes[r, g].axis("off")
axes[0, 0].set_ylabel("Original"); axes[1, 0].set_ylabel("Procesada")
fig.text(0.02, 0.72, "Original", rotation=90, va="center", fontsize=9)
fig.text(0.02, 0.26, "Procesada", rotation=90, va="center", fontsize=9)
plt.tight_layout(rect=[0.035, 0, 1, 1]); plt.savefig(SALIDA/"eda_muestras.png"); plt.show()

## 4. Detección de duplicados (reto RE1)

Se calcula el hash perceptual de cada imagen y se agrupan las que difieren en cuatro bits o menos,
para que cada grupo caiga en un único pliegue. Sin esto, una imagen repetida entre entrenamiento y
validación infla el QWK.

In [ ]:
from aptos.datos import grupos_desde_carpeta

t = time.time()
grupos = grupos_desde_carpeta(etiquetas.id_code, DATOS/"train_images")
n_grupos = int(grupos.nunique())
tam = grupos.value_counts()
n_repetidas = int(tam[tam > 1].sum())
print(f"Calculado en {time.time()-t:.0f} s")
print(f"Grupos distintos: {n_grupos} sobre {len(etiquetas)} imágenes")
print(f"Imágenes que comparten grupo con otra: {n_repetidas} ({100*n_repetidas/len(etiquetas):.1f} %)")
print(f"Grupo más grande: {tam.max()} imágenes")

# Guarda contra sobreagrupación: las fotos de fondo de ojo comparten estructura global,
# así que una tolerancia alta puede unir imágenes que no son duplicadas.
if tam.max() > 20 or n_repetidas > 0.15 * len(etiquetas):
    print("\n*** AVISO: la agrupación parece demasiado amplia. Repite esta celda con "
          "grupos_desde_carpeta(..., distancia_max=2) y compara. Un valor razonable deja "
          "grupos de 2 o 3 imágenes y menos del 10 % agrupado. ***")
else:
    print("\nLa agrupación es conservadora: se puede continuar.")

## 5. Particiones estratificadas

In [ ]:
from aptos.datos import asignar_pliegues

df = asignar_pliegues(etiquetas, cfg["pliegues"], grupos=grupos)
print(pd.crosstab(df.pliegue, df.diagnosis, margins=True).to_string())

fugas = df.groupby(grupos.values).pliegue.nunique()
assert (fugas == 1).all(), "Hay grupos de duplicados repartidos entre pliegues"
print("\nVerificado: ningún grupo de duplicados queda repartido entre pliegues.")

## 6. Caché de imágenes preprocesadas (reto RE2)

Se aplica el recorte circular, el redimensionamiento y la normalización de Graham una sola vez por imagen.
Tarda entre 6 y 10 minutos y evita repetir el cálculo en cada época y en cada pliegue.

In [ ]:
from aptos.entrenar import construir_cache, fijar_semilla

fijar_semilla(cfg["semilla"])
t = time.time()
rutas = construir_cache(df, DATOS/"train_images", Path("/kaggle/working/cache"), cfg["tamano"])
print(f"Caché de {len(rutas)} imágenes construida en {(time.time()-t)/60:.1f} min")

## 7. Entrenamiento

Cada pliegue ejecuta dos épocas con el extractor congelado y hasta `epocas_ajuste` de ajuste fino,
con parada temprana por el QWK de validación. Imprime el QWK de cada pliegue al terminarlo.

In [ ]:
import mlflow
from aptos.entrenar import entrenar_pliegue

dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
oof = np.full(len(df), np.nan); qwks = {}

mlflow.set_tracking_uri("file:/kaggle/working/mlruns")
mlflow.set_experiment("aptos2019")
with mlflow.start_run(run_name=f"{cfg['arquitectura']}-kaggle"):
    mlflow.log_params(cfg)
    for k in PLIEGUES_A_ENTRENAR:
        t = time.time()
        va, pred, q = entrenar_pliegue(k, df, rutas, cfg, dispositivo)
        oof[va] = pred; qwks[k] = q
        print(f"pliegue {k}: QWK = {q:.4f}   ({(time.time()-t)/60:.1f} min)")
    print(f"\nMedia de los pliegues entrenados: {np.mean(list(qwks.values())):.4f}")

## 8. Ajuste de umbrales y resultados

In [ ]:
import json
from aptos.umbrales import OptimizadorUmbrales, a_grados, qwk
from sklearn.metrics import confusion_matrix, f1_score

mask = ~np.isnan(oof)
opt = OptimizadorUmbrales().fit(oof[mask], df.diagnosis[mask])
grados = opt.predict(oof[mask])
reales = df.diagnosis[mask].values

res = {
    "pliegues_entrenados": list(map(int, PLIEGUES_A_ENTRENAR)),
    "qwk_por_pliegue": {str(k): float(v) for k, v in qwks.items()},
    "qwk_media": float(np.mean(list(qwks.values()))),
    "qwk_desv": float(np.std(list(qwks.values()))),
    "qwk_umbrales_iniciales": float(qwk(reales, a_grados(oof[mask]))),
    "qwk_oof_umbrales": float(qwk(reales, grados)),
    "f1_macro": float(f1_score(reales, grados, average="macro")),
    "umbrales": opt.umbrales_.tolist(),
    "n_evaluadas": int(mask.sum()),
    "duplicados": {"grupos": n_grupos, "imagenes_agrupadas": n_repetidas},
    "epocas_ajuste": cfg["epocas_ajuste"],
}

# Sensibilidad y especificidad para RD referible (grado >= 2)
ref_r, ref_p = (reales >= 2), (grados >= 2)
res["sensibilidad_referible"] = float((ref_p & ref_r).sum() / max(ref_r.sum(), 1))
res["especificidad_referible"] = float((~ref_p & ~ref_r).sum() / max((~ref_r).sum(), 1))

json.dump(res, open(SALIDA/"resultados.json", "w"), indent=1, ensure_ascii=False)
print(json.dumps(res, indent=1, ensure_ascii=False))

cm = confusion_matrix(reales, grados, labels=range(5))
print("\nMatriz de confusión (filas = real, columnas = predicho):")
print(pd.DataFrame(cm, index=[f"real {i}" for i in range(5)], columns=[f"pred {i}" for i in range(5)]).to_string())
np.savetxt(SALIDA/"matriz_confusion.csv", cm, fmt="%d", delimiter=",")

## 9. Resumen

In [ ]:
print(f"Tiempo total de la sesión: {(time.time()-T0)/60:.1f} min\n")
print("Archivos generados en /kaggle/working (pestaña Output):")
for p in sorted(SALIDA.glob("*")):
    if p.is_file() and p.suffix in {".json", ".png", ".csv", ".pt"}:
        print(f"  {p.name:28s} {p.stat().st_size/1024:8.1f} KB")
print("\nDescarga resultados.json, las figuras eda_*.png y matriz_confusion.csv, y adjúntalos en el chat.")